# 13 — Application Architecture & Multipage Apps

## 📓 Interactive Notebook · Module 09 · Advanced

In this notebook, you'll learn:
1. **How to evolve** from single-file to modular applications
2. **Separating concerns:** UI, data, and business logic
3. **Creating multipage apps** with `st.navigation` and `st.Page`
4. **Shared components** and configuration
5. **Testing** Streamlit applications

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Structure applications from beginner to advanced
- Create multipage apps with proper navigation
- Implement shared UI components
- Test data logic independently
- Follow naming conventions and best practices

## 📋 Prerequisites

- Modules 01–08 completed
- Understanding of Python modules and imports
- Familiarity with Streamlit widgets and layout

---

## 💡 Level 1: Single File (Beginner)

Every Streamlit app starts as a single file. This is fine for learning and prototypes.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.header("📝 Level 1: Single File App")
st.caption("Everything in one file — simple but doesn't scale")

# Data loading
@st.cache_data
def load_data():
    np.random.seed(42)
    return pd.DataFrame({
        "category": np.random.choice(["A", "B", "C"], 100),
        "value": np.random.randn(100) * 10 + 50
    })

df = load_data()

# UI
category = st.selectbox("Filter", ["All"] + df["category"].unique().tolist())
if category != "All":
    df = df[df["category"] == category]

st.dataframe(df)
st.line_chart(df["value"])

st.info("💡 This works for small apps, but gets messy at 300+ lines.")

---

## 💡 Level 2: Functions (Intermediate)

Extract logic into reusable functions for better organization.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.header("📝 Level 2: Functions")
st.caption("Extract logic into reusable functions")

# --- Data Functions ---
@st.cache_data
def load_data(n_rows):
    """Load data — isolated data concern."""
    np.random.seed(42)
    return pd.DataFrame({
        "category": np.random.choice(["A", "B", "C", "D"], n_rows),
        "value": np.random.randn(n_rows) * 10 + 50,
        "quantity": np.random.randint(1, 100, n_rows)
    })

def filter_data(df, category):
    """Filter data — isolated processing concern."""
    if category == "All":
        return df
    return df[df["category"] == category]

def compute_metrics(df):
    """Compute summary metrics — isolated business logic."""
    return {
        "rows": len(df),
        "categories": df["category"].nunique(),
        "avg_value": df["value"].mean()
    }

# --- UI Functions ---
def render_header():
    """Render page header — isolated UI concern."""
    st.subheader("Dashboard")

def render_metrics(metrics):
    """Render metric cards."""
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Rows", metrics["rows"])
    with col2:
        st.metric("Categories", metrics["categories"])
    with col3:
        st.metric("Avg Value", f"{metrics['avg_value']:.1f}")

def render_data_table(df):
    """Render standardized data table."""
    st.dataframe(df, use_container_width=True)

def render_chart(df):
    """Render visualization."""
    st.line_chart(df["value"])

# --- Main Flow ---
render_header()

category = st.selectbox("Filter by category", ["All"] + ["A", "B", "C", "D"])

df = load_data(200)
filtered = filter_data(df, category)
metrics = compute_metrics(filtered)

render_metrics(metrics)
render_data_table(filtered)
render_chart(filtered)

st.success("✅ Each function has a single responsibility — easier to test and maintain.")

---

## 💡 Level 3: Reusable Components

Create reusable UI components that can be shared across pages.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.header("📝 Reusable Components")
st.caption("Build once, use everywhere")

# --- Reusable Component Functions ---
def metric_card(label, value, delta=None):
    """Reusable metric card with consistent styling."""
    st.metric(label=label, value=value, delta=delta)

def data_preview(df, max_rows=5):
    """Standard data preview with shape info."""
    st.info(f"📊 {df.shape[0]} rows × {df.shape[1]} columns")
    st.dataframe(df.head(max_rows), use_container_width=True)

def section_header(title, description=""):
    """Render section header with optional description."""
    st.subheader(title)
    if description:
        st.caption(description)

def action_button(label, help_text=""):
    """Styled action button."""
    return st.button(label, help=help_text, type="primary")

# --- Using Components ---
section_header("Sales Overview", "Key metrics for the current period")

col1, col2, col3, col4 = st.columns(4)
with col1:
    metric_card("Revenue", "$1.2M", "+12%")
with col2:
    metric_card("Orders", "3,450", "+8%")
with col3:
    metric_card("Customers", "892", "+5%")
with col4:
    metric_card("Avg Order", "$347", "-2%")

section_header("Recent Orders", "Latest transactions")

np.random.seed(42)
orders = pd.DataFrame({
    "Order ID": range(1001, 1006),
    "Customer": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "Amount": [120, 250, 89, 340, 175],
    "Status": ["Complete", "Pending", "Complete", "Shipped", "Complete"]
})

data_preview(orders)

st.info("💡 Components ensure consistency and reduce duplication across pages.")

---

## 💡 Configuration Module

Centralize settings and constants.

In [ ]:
import streamlit as st

st.header("📝 Configuration Pattern")
st.caption("Single source of truth for settings")

# In a real app, this would be config.py
# Here we demonstrate the pattern inline

CONFIG = {
    "app_title": "My Data App",
    "app_icon": "📊",
    "max_rows": 10000,
    "cache_ttl": 3600,
    "default_category": "All",
    "categories": ["Electronics", "Clothing", "Food", "Books"],
}

st.write("**Current Configuration:**")
st.json(CONFIG)

st.write("**Using config values:**")
st.write(f"- App: {CONFIG['app_title']} {CONFIG['app_icon']}")
st.write(f"- Max rows: {CONFIG['max_rows']}")
st.write(f"- Cache TTL: {CONFIG['cache_ttl']}s")
st.write(f"- Categories: {', '.join(CONFIG['categories'])}")

st.info("💡 Config centralizes values — change once, update everywhere.")

---

## 💡 Multipage Apps with `st.navigation`

The modern way to create multipage Streamlit apps.

In [ ]:
import streamlit as st

st.header("📝 Multipage App Pattern")
st.caption("Using st.navigation and st.Page")

st.code('''
# app.py — Entry point
import streamlit as st

# Define pages
pg = st.navigation([
    st.Page("pages/home.py", title="Home", icon="🏠"),
    st.Page("pages/explore.py", title="Explore", icon="📊"),
    st.Page("pages/analysis.py", title="Analysis", icon="🔬"),
])

# Run selected page
pg.run()
''', language="python")

st.write("**Key concepts:**")
- "**`st.Page`** defines a page (Python file or callable)"
- "**`st.navigation`** configures the available pages"
- "**`.run()`** executes the selected page"
- "Entry point acts as a **router** between pages"

st.write("**Grouped navigation:**")
st.code('''
pg = st.navigation({
    "Data": [
        st.Page("pages/home.py", title="Home"),
        st.Page("pages/explore.py", title="Explore"),
    ],
    "Analysis": [
        st.Page("pages/analysis.py", title="Analysis"),
        st.Page("pages/reports.py", title="Reports"),
    ]
})
''', language="python")

---

## 🔬 Experiment: Project Structure Comparison



In [ ]:
import streamlit as st

st.header("🔬 Experiment: Structure Comparison")

col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("🟢 Small")
    st.code('''
my_app/
├── app.py
├── requirements.txt
└── data/
    └── data.csv
''', language="text")
    st.caption("< 300 lines")

with col2:
    st.subheader("🟡 Medium")
    st.code('''
my_app/
├── app.py
├── config.py
├── pages/
│   ├── home.py
│   └── explore.py
├── utils/
│   └── helpers.py
└── data/
''', language="text")
    st.caption("300-1000 lines")

with col3:
    st.subheader("🔴 Large")
    st.code('''
my_app/
├── app.py
├── config.py
├── pages/
├── components/
├── data/
│   ├── loader.py
│   └── processor.py
├── utils/
└── tests/
''', language="text")
    st.caption("1000+ lines")

st.divider()

st.write("**When to level up:**")
st.write("- Single file → Functions: when app exceeds ~100 lines")
st.write("- Functions → Modules: when app exceeds ~500 lines")
st.write("- Modules → Multipage: when app has distinct user workflows")

---

## ⚠️ Common Architectural Mistakes



In [ ]:
import streamlit as st

st.header("⚠️ Common Mistakes")

st.subheader("Mistake 1: Giant app.py")
st.write("❌ Putting everything in one 2000-line file")
st.write("✅ Split into modules at 300-500 lines")

st.subheader("Mistake 2: Business Logic in UI")
st.code('''
# ❌ BAD: Logic mixed with UI
if st.button("Calculate"):
    result = price * quantity * (1 - discount)  # Business logic!
    tax = result * 0.1  # More logic!
    st.write(f"Total: ${result + tax}")

# ✅ GOOD: Separate concerns
def calculate_total(price, qty, discount, tax_rate=0.1):
    subtotal = price * qty * (1 - discount)
    return subtotal * (1 + tax_rate)

if st.button("Calculate"):
    total = calculate_total(price, quantity, discount)
    st.write(f"Total: ${total:.2f}")
''', language="python")

st.subheader("Mistake 3: No Shared Configuration")
st.code('''
# ❌ BAD: Hardcoded paths everywhere
df1 = pd.read_csv("data/sales_2024.csv")  # In file1.py
df2 = pd.read_csv("data/sales_2024.csv")  # In file2.py

# ✅ GOOD: config.py
DATA_PATH = "data/sales_2024.csv"
''', language="python")

st.subheader("Mistake 4: Star Imports")
st.code('''
# ❌ BAD: Pollutes namespace
from utils.helpers import *

# ✅ GOOD: Explicit imports
from utils.helpers import format_currency, validate_email
''', language="python")

---

## 🎯 Challenges

### Challenge 1: Refactor to Functions
Take a monolithic app and extract:
- Data loading function
- Filter function
- UI rendering functions

### Challenge 2: Create Reusable Components
Build a library of:
- Metric card component
- Data table component
- Chart component

### Challenge 3: Design Multipage Structure
Plan a multipage app with:
- Home page
- Data exploration page
- Analysis page
- Settings page

In [ ]:
# Challenge 1: Refactor this code
import streamlit as st
import pandas as pd
import numpy as np

st.write("TODO: Refactor this into separate functions")

# Currently messy code:
# np.random.seed(42)
# df = pd.DataFrame({
#     "category": np.random.choice(["A", "B", "C"], 100),
#     "value": np.random.randn(100) * 10
# })
# cat = st.selectbox("Filter", ["All"] + df["category"].unique().tolist())
# if cat != "All":
#     df = df[df["category"] == cat]
# st.metric("Rows", len(df))
# st.dataframe(df)


---

## 📝 Key Takeaways

1. **Evolution path:** Single file → Functions → Modules → Multipage

2. **Separation of concerns:** UI, Data, Business Logic

3. **`st.navigation` + `st.Page`** is the preferred multipage approach

4. **Reusable components** ensure consistency and reduce duplication

5. **Configuration modules** centralize settings

6. **Test business logic** independently from Streamlit UI

7. **When to level up:** Split when code becomes hard to navigate or maintain

---

## 📚 Further Reading

- [Multipage Apps Overview](https://docs.streamlit.io/develop/concepts/multipage-apps/overview)
- [st.navigation](https://docs.streamlit.io/develop/api-reference/navigation/st.navigation)
- [st.Page](https://docs.streamlit.io/develop/api-reference/navigation/st.Page)

---

## 🔗 Related Materials

- 📖 Reading: [13 — Application Architecture](../readings/13_application_architecture.md)
- ✏️ Exercise: [13 — Architecture Workshop](../exercises/13_architecture_workshop.py)
- 🖥️ Demo App: [13 — Modular App](../apps/13_modular_app/app.py)
- 📝 Quiz: [09 — Architecture & Multipage](../quizzes/09_architecture_multipage.md)